In [1]:
import subprocess, sys

for mod, pkg in [('lightgbm', 'lightgbm>=4.0')]:
    try:
        __import__(mod)
        print(f'{mod} disponible ✅')
    except ImportError:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'{pkg} instalado ✅ — REINICIAR KERNEL antes de continuar')

Instalando lightgbm>=4.0...
lightgbm>=4.0 instalado ✅ — REINICIAR KERNEL antes de continuar



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [1]:

import os, gc, warnings, pickle, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.metrics import (
    f1_score, balanced_accuracy_score,
    confusion_matrix, cohen_kappa_score
)
import mlflow

warnings.filterwarnings('ignore')

SAVE_DIR  = '/srv/work'
IN_FEAT   = f'{SAVE_DIR}/te_features_v6.parquet'
OUT_DIR   = f'{SAVE_DIR}/models'

ROUTE_KEYS = ['source_agent', 'cloud_provider', 'region']
ID_COLS    = ROUTE_KEYS + ['window']
TARGET_COL = 'label_ventana'
SPLIT_COL  = 'split'
HORIZON    = 2
LABEL_COLS = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}

UMBRALES = {
    'qwk':         {'floor': 0.65,  'target': 0.72, 'stretch': 0.80},
    'recall_alta': {'floor': 0.55,  'target': 0.75, 'stretch': 0.85},
    'macro_f1':    {'floor': 0.55},
    'bal_acc':     {'floor': 0.60},
}

W_RF  = 0.80
W_LGB = 0.20
W_XGB = 0.00

os.makedirs(OUT_DIR, exist_ok=True)
print('✅ Config cargada')
print(f'   CANDIDATO: Ensemble D38 — RF×{W_RF} + LGB×{W_LGB} (XGB×{W_XGB})')
print(f'   FLOOR H2 : QWK≥{UMBRALES["qwk"]["floor"]} AND RecAlta≥{UMBRALES["recall_alta"]["floor"]}')
print(f'   ONE-SHOT : Sep 8–Sep 30, 2026 · te_features_v6.parquet')

✅ Config cargada
   CANDIDATO: Ensemble D38 — RF×0.8 + LGB×0.2 (XGB×0.0)
   FLOOR H2 : QWK≥0.65 AND RecAlta≥0.55
   ONE-SHOT : Sep 8–Sep 30, 2026 · te_features_v6.parquet


In [2]:

print('Cargando te_features_v6.parquet...')
df = pq.read_table(IN_FEAT).to_pandas()
print(f'  Total: {len(df):,} ventanas · {df[ROUTE_KEYS[0]].nunique() if ROUTE_KEYS[0] in df else "?"} rutas aprox')

df = df.sort_values(ROUTE_KEYS + ['window'])

df['y'] = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)

with open(f'{OUT_DIR}/feature_cols_v5.json') as f:
    feat_cols_v5 = json.load(f)
print(f'  Features: {len(feat_cols_v5)} (feature_cols_v5.json)')

test_df = df[df[SPLIT_COL] == 'test'].reset_index(drop=True)
y_test   = test_df['y'].values.astype(int)

print(f'\nTEST split: {len(test_df):,} ventanas')
dist = pd.Series(y_test).value_counts(normalize=True).sort_index()
for cls, pct in dist.items():
    print(f'  Clase {cls} ({LABEL_COLS[cls]:6s}): {pct:.1%}')

del df
gc.collect()
print('\n✅ TEST split extraído')

Cargando te_features_v6.parquet...
  Total: 7,628,801 ventanas · 45 rutas aprox
  Features: 100 (feature_cols_v5.json)

TEST split: 1,136,290 ventanas
  Clase 0 (Normal): 82.4%
  Clase 1 (Baja  ): 12.5%
  Clase 2 (Media ): 3.0%
  Clase 3 (Alta  ): 2.1%

✅ TEST split extraído


In [3]:

ratio_idx   = [i for i, c in enumerate(feat_cols_v5) if 'ratio' in c]
std_bgp_idx = [i for i, c in enumerate(feat_cols_v5)
               if c.startswith('std_') or c.startswith('bgp_')]
other_idx   = [i for i in range(len(feat_cols_v5))
               if i not in ratio_idx and i not in std_bgp_idx]

print(f'Índices NaN D17-v5:')
print(f'  ratio_idx  : {len(ratio_idx):3d} features → RF:1.0')
print(f'  std_bgp_idx: {len(std_bgp_idx):3d} features → RF:0.0  LGB:0.0')
print(f'  other_idx  : {len(other_idx):3d} features → RF:0.0  LGB:NaN nativo')

X_test_rf = test_df[feat_cols_v5].values.astype('float32')
for idx in ratio_idx:
    m = np.isnan(X_test_rf[:, idx])
    X_test_rf[m, idx] = 1.0
for idx in std_bgp_idx + other_idx:
    m = np.isnan(X_test_rf[:, idx])
    X_test_rf[m, idx] = 0.0

with open(f'{OUT_DIR}/scaler_v5.pkl', 'rb') as f:
    scaler = pickle.load(f)
X_test_rf_sc = scaler.transform(X_test_rf).astype('float32')
del X_test_rf
gc.collect()
print('\n✅ X_test_rf_sc preparado')

X_test_lgb = test_df[feat_cols_v5].values.astype('float32')
for idx in std_bgp_idx:
    m = np.isnan(X_test_lgb[:, idx])
    X_test_lgb[m, idx] = 0.0
print('✅ X_test_lgb preparado (lags mantienen NaN nativo)')

np.save(f'{OUT_DIR}/X_test_rf_v7.npy',  X_test_rf_sc)
np.save(f'{OUT_DIR}/X_test_lgb_v7.npy', X_test_lgb)
np.save(f'{OUT_DIR}/y_test_v7.npy',     y_test)
print('✅ Arrays guardados: X_test_rf_v7, X_test_lgb_v7, y_test_v7')

Índices NaN D17-v5:
  ratio_idx  :   1 features → RF:1.0
  std_bgp_idx:   6 features → RF:0.0  LGB:0.0
  other_idx  :  93 features → RF:0.0  LGB:NaN nativo

✅ X_test_rf_sc preparado
✅ X_test_lgb preparado (lags mantienen NaN nativo)
✅ Arrays guardados: X_test_rf_v7, X_test_lgb_v7, y_test_v7


In [4]:

REFS = {
    'rf_v51_val':   {'qwk': 0.6503, 'recall_alta': 0.5829, 'split': 'VAL',  'nota': 'v5 val Aug16-Sep7'},
    'ens_v9_val':   {'qwk': 0.6529, 'recall_alta': 0.6509, 'split': 'VAL',  'nota': 'v5 val Aug16-Sep7'},
    'ens_v9_test5': {'qwk': 0.6381, 'recall_alta': 0.5954, 'split': 'TEST', 'nota': 'v5 test 2026-08-06'},
}

def evaluar(nombre, y_true, y_prob, split='TEST', verbose=True):
    y_pred = np.argmax(y_prob, axis=1)

    qwk   = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    mf1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bac   = balanced_accuracy_score(y_true, y_pred)
    mae_o = float(np.mean(np.abs(y_true - y_pred)))
    cm    = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])

    n_alta_true = cm[3].sum()
    n_alta_pred = cm[:, 3].sum()
    rec_a  = float(cm[3, 3] / n_alta_true) if n_alta_true > 0 else 0.0
    prec_a = float(cm[3, 3] / n_alta_pred) if n_alta_pred > 0 else 0.0

    cumple = qwk >= UMBRALES['qwk']['floor'] and rec_a >= UMBRALES['recall_alta']['floor']

    if verbose:
        sep       = '=' * 64
        qwk_icon  = 'OK' if qwk  >= UMBRALES['qwk']['floor']          else 'NO'
        reca_icon = 'OK' if rec_a >= UMBRALES['recall_alta']['floor']  else 'NO'
        mf1_icon  = 'OK' if mf1   >= UMBRALES['macro_f1']['floor']     else 'NO'
        bac_icon  = 'OK' if bac   >= UMBRALES['bal_acc']['floor']      else 'NO'
        cumple_str = 'H2 CUMPLE' if cumple else 'H2 NO CUMPLE'
        print()
        print(sep)
        print(f'  {nombre}')
        print(f'  Split: {split}')
        print(sep)
        print(f'  QWK         : {qwk:.4f}  ({qwk_icon} floor={UMBRALES["qwk"]["floor"]})')
        print(f'  RecAlta     : {rec_a:.4f}  ({reca_icon} floor={UMBRALES["recall_alta"]["floor"]})')
        print(f'  MacroF1     : {mf1:.4f}  ({mf1_icon} floor={UMBRALES["macro_f1"]["floor"]})')
        print(f'  BalAcc      : {bac:.4f}  ({bac_icon} floor={UMBRALES["bal_acc"]["floor"]})')
        print(f'  PrecAlta    : {prec_a:.4f}')
        print(f'  MAE_ord     : {mae_o:.4f}')
        print(f'  H2 criterio : {cumple_str} (QWK>=0.65 AND RecAlta>=0.55)')
        print('  Matriz de confusion [Normal/Baja/Media/Alta]:')
        for idx, row in enumerate(cm):
            print(f'    {LABEL_COLS[idx]:6s}: {row}')

    return dict(
        nombre=nombre, split=split,
        qwk=qwk, macro_f1=mf1, bal_acc=bac, mae_ord=mae_o,
        recall_alta=rec_a, prec_alta=prec_a,
        candidato=cumple, cm=cm
    )

print('Funcion evaluar() lista')
print(f'  val v9 -> RF v5.1: QWK={REFS["rf_v51_val"]["qwk"]:.4f}  '
      f'Ens D38: QWK={REFS["ens_v9_val"]["qwk"]:.4f} RecAlta={REFS["ens_v9_val"]["recall_alta"]:.4f}')
print(f'  test v5 -> Ens D38: QWK={REFS["ens_v9_test5"]["qwk"]:.4f} '
      f'RecAlta={REFS["ens_v9_test5"]["recall_alta"]:.4f} (2026-08-06)')


Funcion evaluar() lista
  val v9 -> RF v5.1: QWK=0.6503  Ens D38: QWK=0.6529 RecAlta=0.6509
  test v5 -> Ens D38: QWK=0.6381 RecAlta=0.5954 (2026-08-06)


In [5]:

print('Cargando RF v5.1 (rf_v51_model.pkl)...')
with open(f'{OUT_DIR}/rf_v51_model.pkl', 'rb') as f:
    rf_v51 = pickle.load(f)
print(f'  Estimators: {rf_v51.n_estimators}  '
      f'class_weight: {rf_v51.class_weight}  '
      f'max_features: {rf_v51.max_features}')

CHUNK = 50_000
n_test = X_test_rf_sc.shape[0]
proba_rf_test = np.zeros((n_test, 4), dtype='float32')

print(f'Prediciendo en chunks de {CHUNK:,}...')
for i in range(0, n_test, CHUNK):
    chunk_end = min(i + CHUNK, n_test)
    proba_rf_test[i:chunk_end] = rf_v51.predict_proba(
        X_test_rf_sc[i:chunk_end]
    ).astype('float32')
    if (i // CHUNK) % 5 == 0:
        print(f'  Chunk {i:>10,} – {chunk_end:>10,} ✓')

np.save(f'{OUT_DIR}/rf_v51_test_v7_proba.npy', proba_rf_test)
print('✅ rf_v51_test_v7_proba.npy guardado')

del rf_v51
gc.collect()

res_rf = evaluar('RF v5.1 — D37 (balanced_subsample, Strobl 2007)', y_test, proba_rf_test)

Cargando RF v5.1 (rf_v51_model.pkl)...
  Estimators: 300  class_weight: balanced_subsample  max_features: sqrt
Prediciendo en chunks de 50,000...
  Chunk          0 –     50,000 ✓
  Chunk    250,000 –    300,000 ✓
  Chunk    500,000 –    550,000 ✓
  Chunk    750,000 –    800,000 ✓
  Chunk  1,000,000 –  1,050,000 ✓
✅ rf_v51_test_v7_proba.npy guardado

  RF v5.1 — D37 (balanced_subsample, Strobl 2007)
  Split: TEST
  QWK         : 0.6737  (OK floor=0.65)
  RecAlta     : 0.5026  (NO floor=0.55)
  MacroF1     : 0.6678  (OK floor=0.55)
  BalAcc      : 0.6550  (OK floor=0.6)
  PrecAlta    : 0.4947
  MAE_ord     : 0.1335
  H2 criterio : H2 NO CUMPLE (QWK>=0.65 AND RecAlta>=0.55)
  Matriz de confusion [Normal/Baja/Media/Alta]:
    Normal: [900995  22954   5175   7252]
    Baja  : [ 34116 101257   4853   1728]
    Media : [ 7794  8139 15139  3186]
    Alta  : [ 6959  1603  3227 11913]


In [6]:

print('Cargando LGB v7 (lgb_v7_model.txt)...')
lgb_v7 = lgb.Booster(model_file=f'{OUT_DIR}/lgb_v7_model.txt')
print(f'  Num trees: {lgb_v7.num_trees()}')

print('Prediciendo...')
proba_lgb_test = lgb_v7.predict(X_test_lgb).astype('float32')
np.save(f'{OUT_DIR}/lgb_v7_test_v7_proba.npy', proba_lgb_test)
print('✅ lgb_v7_test_v7_proba.npy guardado')

del lgb_v7
gc.collect()

res_lgb = evaluar('LGB v7 — D33 (sample_weight sqrt, He & Garcia 2009)', y_test, proba_lgb_test)

Cargando LGB v7 (lgb_v7_model.txt)...
  Num trees: 8000
Prediciendo...
✅ lgb_v7_test_v7_proba.npy guardado

  LGB v7 — D33 (sample_weight sqrt, He & Garcia 2009)
  Split: TEST
  QWK         : 0.6614  (OK floor=0.65)
  RecAlta     : 0.4852  (NO floor=0.55)
  MacroF1     : 0.6681  (OK floor=0.55)
  BalAcc      : 0.6763  (OK floor=0.6)
  PrecAlta    : 0.4765
  MAE_ord     : 0.1438
  H2 criterio : H2 NO CUMPLE (QWK>=0.65 AND RecAlta>=0.55)
  Matriz de confusion [Normal/Baja/Media/Alta]:
    Normal: [886242  33010   8568   8556]
    Baja  : [ 28721 105559   6125   1549]
    Media : [ 6476  7096 18159  2527]
    Alta  : [ 6592  1655  3955 11500]


In [7]:

print('='*64)
print('  EVALUACIÓN ONE-SHOT — CANDIDATO DEFINITIVO')
print('  Ensemble D38: RF×0.80 + LGB×0.20')
print('='*64)

proba_ens = (W_RF * proba_rf_test + W_LGB * proba_lgb_test).astype('float32')
np.save(f'{OUT_DIR}/ens_v9_test_v7_proba.npy', proba_ens)
print('✅ ens_v9_test_v7_proba.npy guardado')

res_ens = evaluar(
    'Ensemble D38 — RF v5.1×0.80 + LGB v7×0.20 — CANDIDATO DEFINITIVO',
    y_test, proba_ens
)

delta_qwk = res_ens['qwk'] - REFS['ens_v9_val']['qwk']
print(f'\n  ΔQWK val→test : {delta_qwk:+.4f}  '
      f'(val={REFS["ens_v9_val"]["qwk"]:.4f} → test={res_ens["qwk"]:.4f})')
print(f'  ΔQWK prev test: {res_ens["qwk"] - REFS["ens_v9_test5"]["qwk"]:+.4f}  '
      f'(test_v5={REFS["ens_v9_test5"]["qwk"]:.4f} → test_v7={res_ens["qwk"]:.4f})')

  EVALUACIÓN ONE-SHOT — CANDIDATO DEFINITIVO
  Ensemble D38: RF×0.80 + LGB×0.20
✅ ens_v9_test_v7_proba.npy guardado

  Ensemble D38 — RF v5.1×0.80 + LGB v7×0.20 — CANDIDATO DEFINITIVO
  Split: TEST
  QWK         : 0.6786  (OK floor=0.65)
  RecAlta     : 0.4966  (NO floor=0.55)
  MacroF1     : 0.6763  (OK floor=0.55)
  BalAcc      : 0.6647  (OK floor=0.6)
  PrecAlta    : 0.5091
  MAE_ord     : 0.1316
  H2 criterio : H2 NO CUMPLE (QWK>=0.65 AND RecAlta>=0.55)
  Matriz de confusion [Normal/Baja/Media/Alta]:
    Normal: [899884  24089   5380   7023]
    Baja  : [ 32732 102659   4980   1583]
    Media : [ 7482  7658 16372  2746]
    Alta  : [ 6959  1508  3464 11771]

  ΔQWK val→test : +0.0257  (val=0.6529 → test=0.6786)
  ΔQWK prev test: +0.0405  (test_v5=0.6381 → test_v7=0.6786)


In [8]:

import warnings
warnings.filterwarnings('ignore')

filas = [
    ('Ens D38 val v9 (ref)',        'VAL',  REFS['ens_v9_val']['qwk'],   REFS['ens_v9_val']['recall_alta'],   'Referencia frozen LOG-25'),
    ('Ens D38 test v5 (2026-08-06)','TEST', REFS['ens_v9_test5']['qwk'], REFS['ens_v9_test5']['recall_alta'], 'Evaluación previa (❌ QWK<0.65)'),
    ('RF v5.1   test v7',           'TEST', res_rf['qwk'],               res_rf['recall_alta'],               'D37 solo'),
    ('LGB v7    test v7',           'TEST', res_lgb['qwk'],              res_lgb['recall_alta'],              'D33 solo'),
    ('Ens D38   test v7 ⭐',        'TEST', res_ens['qwk'],              res_ens['recall_alta'],              'CANDIDATO DEFINITIVO D41-6'),
]

hdr  = f"{'Modelo':<40} {'Split':<5} {'QWK':>7} {'RecAlta':>8} {'≥floor':>7}  Nota"
sep  = '-' * len(hdr)
print(f'\n{sep}')
print(hdr)
print(sep)
for nombre, split, qwk, rec, nota in filas:
    ok = '✅' if (qwk >= UMBRALES['qwk']['floor'] and rec >= UMBRALES['recall_alta']['floor']) else '❌'
    print(f'{nombre:<40} {split:<5} {qwk:>7.4f} {rec:>8.4f} {ok:>7}  {nota}')
print(sep)

qwk_ok  = res_ens['qwk']          >= UMBRALES['qwk']['floor']
rec_ok  = res_ens['recall_alta']  >= UMBRALES['recall_alta']['floor']
mf1_ok  = res_ens['macro_f1']     >= UMBRALES['macro_f1']['floor']
bac_ok  = res_ens['bal_acc']      >= UMBRALES['bal_acc']['floor']

print('\n' + '='*64)
print('  VEREDICTO H2 — ONE-SHOT PROTOCOL (Kaufman et al., 2012)')
print('='*64)
print(f'  Criterio principal : QWK≥0.65 AND RecAlta≥0.55 (LOG-25)')
print(f'    QWK        = {res_ens["qwk"]:.4f}  {"✅" if qwk_ok else "❌"}')
print(f'    RecAlta    = {res_ens["recall_alta"]:.4f}  {"✅" if rec_ok else "❌"}')
print(f'  Criterios secundarios:')
print(f'    MacroF1    = {res_ens["macro_f1"]:.4f}  {"✅" if mf1_ok else "⚠️"}')
print(f'    BalAcc     = {res_ens["bal_acc"]:.4f}  {"✅" if bac_ok else "⚠️"}')

if qwk_ok and rec_ok:
    veredicto = '✅ H2 VERIFICADA'
    veredicto_desc = 'QWK y RecAlta superan el floor simultáneamente'
elif qwk_ok or rec_ok:
    veredicto = '⚠️ H2 PARCIALMENTE VERIFICADA'
    fallo = 'QWK' if not qwk_ok else 'RecAlta'
    veredicto_desc = f'Falla criterio conjuntivo: {fallo} por debajo del floor'
else:
    veredicto = '❌ H2 NO VERIFICADA'
    veredicto_desc = 'Ninguna métrica principal supera el floor'

print(f'\n  >> {veredicto}')
print(f'     {veredicto_desc}')
print('='*64)

import pickle as _pkl
resultados = {
    'res_rf':  res_rf,
    'res_lgb': res_lgb,
    'res_ens': res_ens,
    'veredicto': veredicto,
    'refs': REFS,
    'umbrales': UMBRALES,
    'pesos': {'W_RF': W_RF, 'W_LGB': W_LGB, 'W_XGB': W_XGB},
    'split': 'test · Sep 8–Sep 30, 2026 · te_features_v6.parquet',
    'fecha': '2026-10-06',
    'candidato': 'Ensemble D38 — RF v5.1×0.80 + LGB v7×0.20',
    'log': 'D41-6',
}
with open(f'{OUT_DIR}/resultados_test_v7.pkl', 'wb') as f:
    _pkl.dump(resultados, f)
print('\n✅ resultados_test_v7.pkl guardado')

mlflow.set_tracking_uri(f'{SAVE_DIR}/mlruns')
mlflow.set_experiment('latam2cloud_oe2')

with mlflow.start_run(run_name='D41-6_evaluacion_test_v7_one_shot') as run:
    mlflow.set_tags({
        'etapa':     'D41-6',
        'split':     'test_v7',
        'candidato': 'Ensemble_D38',
        'protocolo': 'one_shot',
        'log':       'LOG-50',
        'dataset':   'te_features_v6_parquet',
    })
    mlflow.log_metrics({
        'ens_qwk':       res_ens['qwk'],
        'ens_recall_alta': res_ens['recall_alta'],
        'ens_macro_f1':  res_ens['macro_f1'],
        'ens_bal_acc':   res_ens['bal_acc'],
        'ens_prec_alta': res_ens['prec_alta'],
        'ens_mae_ord':   res_ens['mae_ord'],
        'rf_qwk':        res_rf['qwk'],
        'rf_recall_alta':res_rf['recall_alta'],
        'lgb_qwk':       res_lgb['qwk'],
        'lgb_recall_alta':res_lgb['recall_alta'],
        'delta_qwk_val_test': res_ens['qwk'] - REFS['ens_v9_val']['qwk'],
        'delta_qwk_prev_test': res_ens['qwk'] - REFS['ens_v9_test5']['qwk'],
    })
    mlflow.log_params({
        'W_RF': W_RF, 'W_LGB': W_LGB, 'W_XGB': W_XGB,
        'candidato': 'Ens_D38_RF_v51_LGB_v7',
        'test_split': 'Sep8-Sep30-2026',
        'n_test': len(y_test),
        'veredicto_h2': veredicto,
        'qwk_floor': UMBRALES['qwk']['floor'],
        'recall_alta_floor': UMBRALES['recall_alta']['floor'],
    })
    print(f'\n✅ MLflow run: {run.info.run_id}')

print('\n✅ D41-6 completado')
print(f'   Notebook: 06 evaluacion test set v7.ipynb')
print(f'   Documentar resultado en: MODELING_ANALYSIS.md §30 + §11.2')

2026/10/10 14:37:37 INFO mlflow.tracking.fluent: Experiment with name 'latam2cloud_oe2' does not exist. Creating a new experiment.



-----------------------------------------------------------------------------
Modelo                                   Split     QWK  RecAlta  ≥floor  Nota
-----------------------------------------------------------------------------
Ens D38 val v9 (ref)                     VAL    0.6529   0.6509       ✅  Referencia frozen LOG-25
Ens D38 test v5 (2026-08-06)             TEST   0.6381   0.5954       ❌  Evaluación previa (❌ QWK<0.65)
RF v5.1   test v7                        TEST   0.6737   0.5026       ❌  D37 solo
LGB v7    test v7                        TEST   0.6614   0.4852       ❌  D33 solo
Ens D38   test v7 ⭐                      TEST   0.6786   0.4966       ❌  CANDIDATO DEFINITIVO D41-6
-----------------------------------------------------------------------------

  VEREDICTO H2 — ONE-SHOT PROTOCOL (Kaufman et al., 2012)
  Criterio principal : QWK≥0.65 AND RecAlta≥0.55 (LOG-25)
    QWK        = 0.6786  ✅
    RecAlta    = 0.4966  ❌
  Criterios secundarios:
    MacroF1    = 0.6763  